In [12]:
!pip install langchain langchain-text-splitters

In [9]:
pip install pymupdf

   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
    ----------------

In [16]:
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. LOAD: Extract text from your PDF
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

# Pass the file path here
raw_text = extract_text(r"C:\Users\PC\Downloads\HDFC_Bank_Annual_Report_2024_25-310202.pdf")

# 2. CLEAN
cleaned_text = " ".join(raw_text.split())

# 3. CHUNK
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_text(cleaned_text)

print(f"Total chunks created: {len(chunks)}")
print(f"Example Chunk: {chunks[0][:200]}...")

Total chunks created: 2103
Example Chunk: Powering Progress Together Banking today is more than managing money — it’s about fulﬁlling aspirations, and shaping futures. At HDFC Bank, we see ourselves not just as custodians of capital but as pa...


In [17]:
print(chunks[500])

across channels. With the delivery engine in place, HDFC Bank is now poised to tap into the potential emerging technologies that can shape the future of banking. Looking Ahead: The Next Chapter of AI-Readiness As industries globally embrace the possibilities of Generative AI, the Bank is preparing for a calibrated and structured adoption. HDFC Bank is evaluating the transformative potential of Generative AI (GenAI) to enhance efﬁciencies and customer experiences. It is assessing GenAI capabilities across key processes while ensuring responsible and secure implementation through a platform-driven approach. While still in the exploratory phase, GenAI is expected to play a meaningful role in shaping future-ready banking experiences. Integrated Annual Report 2024-25 149 Responsible Business SOCIAL - PEOPLE In today’s dynamic ﬁnancial landscape, our competitive advantage lies in Strategy, People and Technology. As we look ahead, strengthening leadership, enabling continuous capability


In [19]:
pip install chromadb sentence-transformers

  Using cached chromadb-1.5.5-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached build-1.4.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached pybase64-1.4.3-cp312-cp312-win_amd64.whl.metadata (9.1 kB)
  Using cached uvicorn-0.43.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached onnxruntime-1.24.4-cp312-cp312-win_amd64.whl.metadata (5.4 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.40.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached grpcio-1.80.0-cp31

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.6 which is incompatible.


In [20]:
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Load the "Embedding Model" (The translator that turns text into numbers)
# 'all-MiniLM-L6-v2' is fast, free, and runs on your laptop.
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Initialize ChromaDB (This creates a local database on your PC)
client = chromadb.Client()
collection = client.get_or_create_collection(name="hdfc_banking_bot")

# 3. Add your chunks to the database
# This might take a minute since you have 2,103 chunks!
print("Indexing chunks... please wait.")

for i, chunk in enumerate(chunks):
    # Turn text into a list of numbers (Embedding)
    vector = model.encode(chunk).tolist()
    
    # Store it in the database
    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[vector],
        documents=[chunk],
        metadatas=[{"source": "HDFC Annual Report 2024-25", "page": "varies"}]
    )

print(f"Successfully indexed {len(chunks)} chunks into the Vector DB!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexing chunks... please wait.
Successfully indexed 2103 chunks into the Vector DB!


In [21]:
# Simulate a Relationship Manager's question
query = "What is HDFC Bank's plan for Generative AI?"

# Search the database for the top 2 most relevant chunks
query_vector = model.encode(query).tolist()
results = collection.query(query_embeddings=[query_vector], n_results=2)

print("TOP SEARCH RESULT:")
print(results['documents'][0][0])

TOP SEARCH RESULT:
markets while reimaging best in class customer experiences. Generative AI (GenAI) presents a generational transformation opportunity and responsibility. However, we are ensuring there is adequate appreciation of risk, reliability and information security which is embedded in our approach to designing the cutting-edge AI solutions in a calibrated fashion. Going forward, with our pan India geographical spread and large customer base, I remain conﬁdent in our ability to deliver personalised and secure banking experiences to individuals and enterprises we serve. India’s banking and ﬁntech landscape has already made a strong mark on the global stage – with its scale, innovation and digital adoption setting new benchmarks. This momentum is only set to accelerate and HDFC Bank is committed to playing its part in shaping this future. Social Value Creation Inclusive banking is the cornerstone of Inclusion and is a major policy of the Government. Credit ﬂow to underserved sect

In [24]:
query = "What is HDFC net profit?"
query_vector = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)
print(results)

{'ids': [['chunk_942', 'chunk_831', 'chunk_1871']], 'embeddings': None, 'documents': [['the immovable property as well as boundaries). 9. Specify the reason(s), if the company has failed to spend two per cent of the average net profit as per sub-section (5) of section 135. NA Sashidhar Jagdishan Managing Director and Chief Executive Officer Sunita Maheshwari Chairperson of CSR & ESG Committee HDFC Bank Limited 268 D I R E C T O R S ’ R E P O R T Performance and financial position of the subsidiaries/entity over which control is exercised of the Bank as on March 31, 2025 Name of entity Net assets as of March 31, 2025 Profit or (loss) for the year ended March 31, 2025 As % of consolidated net assets** Amount Rs in crore As % of consolidated profit or loss Amount Rs in crore Parent: HDFC Bank Limited 93.18% 501,424.64 91.70% 67,347.36 Subsidiaries: HDFC Life Insurance Company Limited (Consolidated)*1 3.00% 16,153.93 2.39% 1,757.94 HDB Financial Services Limited* 3.00% 16,142.41 2.88% 2,11

In [25]:
# Assuming you are using an LLM API (like OpenAI or Gemini)
def generate_rm_response(query, context_chunks):
    # Combine the chunks into one string
    context_text = "\n\n".join(context_chunks)
    
    system_prompt = """
    You are a Relationship Manager Support Bot for HDFC Bank. 
    Your goal is to help RMs answer client questions accurately based ONLY on the provided context.
    If the context doesn't contain the answer, politely inform the RM.
    Keep the tone professional, concise, and helpful.
    """
    
    user_prompt = f"""
    Context from Annual Report:
    {context_text}
    
    Question: {query}
    
    Helpful Response for the RM:
    """
    
    # Here you would call your LLM API:
    # response = llm.invoke(system_prompt + user_prompt)
    # return response
    print("PROMPT READY FOR LLM")

# Test it with your previous result
generate_rm_response("What is our stance on GenAI risks?", results['documents'][0])

PROMPT READY FOR LLM
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\PC\\anaconda3\\Lib\\site-packages\\safetensors\\_safetensors_rust.pyd'
Consider using the `--user` option or check the permissions.

